In [ ]:
import sys
sys.path.append('..')
import numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation, metric_fit, mesh
from numpy.linalg import norm

In [ ]:
m, fuseMarkers = wall_generation.triangulate_channel_walls(*parametric_pillows.concentricCircles(1, 60), 0.001)
V = m.vertices()[:, 0:2]
F = m.triangles()

In [ ]:
radialContraction = True
barycenters = np.mean(V[F], axis=1)
contractionDirs = barycenters / np.linalg.norm(barycenters, axis=1)[:, None]
uncontractedDirs = np.column_stack((-contractionDirs[:, 1], contractionDirs[:, 0]))

if (not radialContraction):
    contractionDirs, uncontractedDirs = uncontractedDirs, contractionDirs

g = (np.einsum('ij,ik->ijk', (2 / np.pi)**2 * contractionDirs, contractionDirs) +
     np.einsum('ij,ik->ijk',       1.0 * uncontractedDirs, uncontractedDirs))

In [ ]:
mflat = metric_fit.Mesh2D(V, F)
fitter = metric_fit.MetricFitter(mflat)
fitter.setTargetMetric(g)

In [ ]:
from tri_mesh_viewer import TriMeshViewer
immersedSurface = mesh.Mesh(fitter.getImmersion().transpose(), m.triangles())
import vis
vf = vis.fields.VectorField(np.pad(uncontractedDirs, [(0, 0), (0, 1)], mode='constant'))
viewer = TriMeshViewer(immersedSurface, width=1024, height=640)
viewer.arrowSize = 40
viewer.showWireframe()
viewer.show()

In [ ]:
fitter.bendingStiffness = 1e-5
fitter.setVars(fitter.getVars() + 1e-4 * np.random.uniform(low=-1, high=1, size=fitter.numVars()))
maxDistortion = np.max(np.sqrt(fitter.metricDistSq()))

In [ ]:
fitter.collapsePreventionWeight = 1.0

In [ ]:
fitter.energy(fitter.EnergyType.CollapsePrevention)

In [ ]:
import time, matplotlib
from py_newton_optimizer import NewtonOptimizerOptions

opt = NewtonOptimizerOptions()
opt.gradTol = 1e-9
opt.niter = 10

metric_fit.benchmark_reset()
for i in range(100):
    cr = metric_fit.fit_metric_newton(fitter, fitter.rigidMotionPinVars, opt)
    if len(cr.energy) < 2: break
    immersedSurface = mesh.Mesh(fitter.getImmersion().transpose(), m.triangles())
    sf = vis.fields.ScalarField(np.sqrt(fitter.metricDistSq()), colormap=matplotlib.cm.coolwarm, vmin=0, vmax=maxDistortion)
    viewer.update(mesh=immersedSurface, scalarField=sf)
    time.sleep(0.01)
metric_fit.benchmark_report()